# Chapter 2: Data Access Module

This chapter covers the data access layer for ETF research, including:
- Database connection management
- ETF data loading
- NAV and unit data retrieval
- Bond curve data access
- Data caching strategies

## 1. Setup and Imports

In [ ]:
import datetime
import os
import sys
import warnings

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.pool import QueuePool

# Add project root to path
project_root = os.path.dirname(os.path.abspath('__file__'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import ETF_DB, BOND_DB, setup_logging, DATA_DIR
from utils import DataLoader, get_trading_days, get_latest_trading_day

logger = setup_logging('data_access')

## 2. Database Connection Manager

The `DataLoader` class provides a unified interface for database operations.

In [ ]:
class EnhancedDataLoader(DataLoader):
    """Enhanced data loader with additional utilities"""
    
    def __init__(self, db_config=None):
        super().__init__(db_config)
    
    def get_table_info(self, table_name: str) -> pd.DataFrame:
        """Get table structure information
        
        Args:
            table_name: Full table name (e.g., 'fund.marketinfo')
            
        Returns:
            DataFrame with column information
        """
        sql = """
        SELECT 
            COLUMN_NAME,
            DATA_TYPE,
            IS_NULLABLE,
            COLUMN_KEY,
            COLUMN_COMMENT
        FROM information_schema.COLUMNS
        WHERE TABLE_SCHEMA = DATABASE()
        AND TABLE_NAME = :table_name
        ORDER BY ORDINAL_POSITION
        """
        # Extract table name from full path
        table = table_name.split('.')[-1]
        return self.query_df(sql, (table,))
    
    def get_table_stats(self, table_name: str) -> dict:
        """Get table statistics
        
        Args:
            table_name: Full table name
            
        Returns:
            Dictionary with table statistics
        """
        sql = f"""
        SELECT 
            COUNT(*) as row_count,
            MIN(DT) as min_date,
            MAX(DT) as max_date
        FROM {table_name}
        """
        result = self.query_df(sql)
        return result.iloc[0].to_dict()
    
    def execute_insert(self, table_name: str, df: pd.DataFrame, 
                       if_exists: str = 'append') -> int:
        """Insert DataFrame into table
        
        Args:
            table_name: Target table name
            df: DataFrame to insert
            if_exists: Behavior when table exists
            
        Returns:
            Number of rows inserted
        """
        rows = df.to_sql(
            name=table_name.split('.')[-1],
            con=self.engine,
            if_exists=if_exists,
            index=False,
            method='multi'
        )
        return rows

# Initialize enhanced data loader
loader = EnhancedDataLoader()

## 3. ETF Base Information Access

In [ ]:
# Get ETF base information
def load_etf_base_info(loader: DataLoader, dt: str = None) -> pd.DataFrame:
    """Load ETF base information with data validation
    
    Args:
        loader: DataLoader instance
        dt: Date string, defaults to latest
        
    Returns:
        DataFrame with ETF base info
    """
    df = loader.get_etf_base_info(dt)
    
    if df.empty:
        logger.warning("No ETF base info data found")
        return df
    
    # Data validation
    print(f"ETF Base Info Summary:")
    print(f"- Total ETFs: {len(df)}")
    print(f"- Date: {df['DT'].iloc[0] if 'DT' in df.columns else 'N/A'}")
    print(f"\nColumns: {list(df.columns)}")
    
    # Market direction distribution
    if 'MARKET_DIRECTION' in df.columns:
        print(f"\nMarket Direction Distribution:")
        print(df['MARKET_DIRECTION'].value_counts())
    
    return df

# Load ETF base info
etf_base = load_etf_base_info(loader)
etf_base.head()

## 4. NAV Data Access

In [ ]:
def load_etf_nav_data(loader: DataLoader, codes: list, 
                      start_date: str = None, end_date: str = None) -> pd.DataFrame:
    """Load ETF NAV data with preprocessing
    
    Args:
        loader: DataLoader instance
        codes: List of ETF codes
        start_date: Start date
        end_date: End date
        
    Returns:
        DataFrame with NAV data
    """
    # Get NAV data
    nav_df = loader.get_etf_nav_data(codes, start_date, end_date)
    
    if nav_df.empty:
        logger.warning("No NAV data found")
        return nav_df
    
    # Convert date column
    nav_df['DT'] = pd.to_datetime(nav_df['DT'])
    
    # Sort by code and date
    nav_df = nav_df.sort_values(['TRADE_CODE', 'DT'])
    
    # Forward fill missing values
    nav_df = nav_df.groupby('TRADE_CODE').apply(
        lambda x: x.ffill()
    ).reset_index(drop=True)
    
    # Summary statistics
    print(f"NAV Data Summary:")
    print(f"- Total records: {len(nav_df)}")
    print(f"- ETFs covered: {nav_df['TRADE_CODE'].nunique()}")
    print(f"- Date range: {nav_df['DT'].min()} to {nav_df['DT'].max()}")
    
    return nav_df

# Example: Load NAV data for first 5 ETFs
if not etf_base.empty:
    sample_codes = etf_base['TRADE_CODE'].head(5).tolist()
    start_date, end_date = '2024-01-01', '2024-12-31'
    nav_data = load_etf_nav_data(loader, sample_codes, start_date, end_date)
    print(f"\nSample NAV data:")
    display(nav_data.head(10))

## 5. Unit Data Access

In [ ]:
def load_etf_unit_data(loader: DataLoader, codes: list,
                       start_date: str = None, end_date: str = None) -> pd.DataFrame:
    """Load ETF unit/share data
    
    Args:
        loader: DataLoader instance
        codes: List of ETF codes
        start_date: Start date
        end_date: End date
        
    Returns:
        DataFrame with unit data
    """
    unit_df = loader.get_etf_unit_data(codes, start_date, end_date)
    
    if unit_df.empty:
        logger.warning("No unit data found")
        return unit_df
    
    # Convert date column
    unit_df['DT'] = pd.to_datetime(unit_df['DT'])
    
    # Filter out zero units
    unit_df = unit_df[unit_df['UNIT_ALL'] > 0]
    
    print(f"Unit Data Summary:")
    print(f"- Total records: {len(unit_df)}")
    print(f"- ETFs covered: {unit_df['TRADE_CODE'].nunique()}")
    
    # Calculate total AUM
    total_aum = unit_df['UNIT_ALL'].sum() / 1e8  # Convert to 100 million
    print(f"- Total AUM: {total_aum:.2f} billion")
    
    return unit_df

# Example: Load unit data
if not etf_base.empty:
    unit_data = load_etf_unit_data(loader, sample_codes, start_date, end_date)
    print(f"\nSample unit data:")
    display(unit_data.head(10))

## 6. Bond Curve Data Access

In [ ]:
def load_bond_curve_data(loader: DataLoader, start_date: str, 
                         end_date: str = None, bond_type: str = None) -> pd.DataFrame:
    """Load bond curve data
    
    Args:
        loader: DataLoader instance
        start_date: Start date
        end_date: End date
        bond_type: Bond type filter
        
    Returns:
        DataFrame with bond curve data
    """
    curve_df = loader.get_bond_curve_data(start_date, end_date, bond_type)
    
    if curve_df.empty:
        logger.warning("No bond curve data found")
        return curve_df
    
    print(f"Bond Curve Data Summary:")
    print(f"- Total records: {len(curve_df)}")
    print(f"- Date range: {curve_df['dt'].min()} to {curve_df['dt'].max()}")
    
    if 'bond_type' in curve_df.columns:
        print(f"\nBond types: {curve_df['bond_type'].unique()}")
    
    return curve_df

# Note: Bond curve data requires bond database access
# bond_curve = load_bond_curve_data(loader, '2024-01-01', '2024-12-31')

## 7. Data Quality Check

In [ ]:
def check_data_quality(nav_df: pd.DataFrame, unit_df: pd.DataFrame) -> dict:
    """Check data quality for NAV and unit data
    
    Args:
        nav_df: NAV DataFrame
        unit_df: Unit DataFrame
        
    Returns:
        Dictionary with quality metrics
    """
    quality_report = {}
    
    # NAV data quality
    if not nav_df.empty:
        quality_report['nav'] = {
            'total_records': len(nav_df),
            'missing_nav': nav_df['NAV'].isna().sum(),
            'missing_nav_adj': nav_df['NAV_ADJ'].isna().sum(),
            'zero_nav': (nav_df['NAV'] == 0).sum(),
            'negative_nav': (nav_df['NAV'] < 0).sum()
        }
    
    # Unit data quality
    if not unit_df.empty:
        quality_report['unit'] = {
            'total_records': len(unit_df),
            'missing_unit': unit_df['UNIT_ALL'].isna().sum(),
            'zero_unit': (unit_df['UNIT_ALL'] == 0).sum(),
            'negative_unit': (unit_df['UNIT_ALL'] < 0).sum()
        }
    
    # Print report
    print("Data Quality Report")
    print("=" * 50)
    
    for data_type, metrics in quality_report.items():
        print(f"\n{data_type.upper()} Data:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value}")
    
    return quality_report

# Check data quality
if 'nav_data' in dir() and 'unit_data' in dir():
    quality = check_data_quality(nav_data, unit_data)

## 8. Data Export Utilities

In [ ]:
def export_data(df: pd.DataFrame, filename: str, format: str = 'csv'):
    """Export DataFrame to file
    
    Args:
        df: DataFrame to export
        filename: Output filename (without extension)
        format: Export format ('csv', 'excel', 'parquet')
    """
    output_path = DATA_DIR / filename
    
    if format == 'csv':
        df.to_csv(f"{output_path}.csv", index=False, encoding='utf-8-sig')
    elif format == 'excel':
        df.to_excel(f"{output_path}.xlsx", index=False)
    elif format == 'parquet':
        df.to_parquet(f"{output_path}.parquet", index=False)
    else:
        raise ValueError(f"Unsupported format: {format}")
    
    logger.info(f"Data exported to {output_path}.{format}")

# Example export
# export_data(nav_data, 'etf_nav_sample', 'csv')

## 9. Summary

This chapter covered:
1. Database connection management using DataLoader
2. ETF base information loading
3. NAV data retrieval and preprocessing
4. Unit/share data access
5. Bond curve data access
6. Data quality checking
7. Data export utilities

### Key Classes
- `DataLoader`: Main data access class
- `EnhancedDataLoader`: Extended functionality

### Best Practices
- Always check for empty dataframes
- Use connection pooling for performance
- Validate data quality before analysis
- Handle missing values appropriately